# ML 04 — Grouped validation design and leakage-safe preprocessing

This notebook creates explicit family-held-out and layout-held-out folds from the engineered dataset. The **primary** estimate holds out complete plan families because the brief asks for generalization to unseen geometries. Layout-held-out validation is a less strict secondary check because other layouts from the same family can appear in training. Every imputer, scaler and encoder is fitted inside the training fold via a scikit-learn Pipeline.

In [ ]:
from pathlib import Path
import os

# Works whether Jupyter starts in the repository root or this notebook's folder.
_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "data" / "raw").exists() and (p / "Machine Learning Pipeline").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find the repository root. Open this notebook from inside the T006 project folder.")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

processed_path = DATA_PROCESSED / "feature_engineered_dataset.csv"
if not processed_path.exists():
    raise FileNotFoundError("Run ML_01, then ML_03 first; ML_03 builds the engineered modelling dataset from raw data.")
df = pd.read_csv(processed_path)
TARGET = "target_U_canopy_mean_m_s"
NUMERIC_FEATURES = ["plan_area_density", "plan_area_density_sq", "height_std_m", "height_max_m", "height_min_m", "height_range_m", "height_skew", "frac_taller_than_mean", "wind_direction_sin", "wind_direction_cos", "height_coefficient_variation"]
CATEGORICAL_FEATURES = ["configuration", "horizontal_configuration", "density_class"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
BASELINE_NUMERIC = ["plan_area_density", "height_std_m", "height_max_m", "building_count", "wind_direction_sin", "wind_direction_cos"]
BASELINE_FEATURES = BASELINE_NUMERIC + CATEGORICAL_FEATURES
print("Dataset shape:", df.shape)
print("Target:", TARGET)

## 1. Primary protocol: hold out complete plan families

A group split is required because multiple rows share a family and each layout appears under two wind directions. In GroupKFold, every family is assigned to either train or test within a fold. Report the number of groups and rows per fold; fold scores may differ because only 16 families are available.

In [ ]:
from sklearn.model_selection import GroupKFold

X = df[FEATURES]
y = df[TARGET]
groups = df["family_id"]
cv = GroupKFold(n_splits=5)
fold_rows = []
for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
    train_groups = set(groups.iloc[train_idx])
    test_groups = set(groups.iloc[test_idx])
    assert train_groups.isdisjoint(test_groups)
    assert set(df.iloc[train_idx]["layout_id"]).isdisjoint(set(df.iloc[test_idx]["layout_id"]))
    fold_rows.append({
        "protocol": "family_5fold", "fold": fold,
        "train_cases": len(train_idx), "test_cases": len(test_idx),
        "train_families": len(train_groups), "test_families": len(test_groups),
        "train_family_ids": ",".join(sorted(train_groups)),
        "test_family_ids": ",".join(sorted(test_groups)),
    })
fold_table = pd.DataFrame(fold_rows)
display(fold_table.drop(columns=["train_family_ids", "test_family_ids"]))
print("Family leakage check passed for every fold.")

## 2. Secondary protocol: hold out layouts

This protocol tests transfer to layouts not seen during training, but it may still train on other layouts from the same family. Therefore it is an interpolation-like check and should not replace family-held-out scores as the primary generalization estimate.

In [ ]:
layout_cv = GroupKFold(n_splits=5)
layout_rows = []
for fold, (train_idx, test_idx) in enumerate(layout_cv.split(X, y, df["layout_id"]), start=1):
    train_layouts = set(df.iloc[train_idx]["layout_id"])
    test_layouts = set(df.iloc[test_idx]["layout_id"])
    assert train_layouts.isdisjoint(test_layouts)
    shared_families = set(df.iloc[train_idx]["family_id"]) & set(df.iloc[test_idx]["family_id"])
    layout_rows.append({"fold": fold, "train_cases": len(train_idx), "test_cases": len(test_idx), "test_layouts": len(test_layouts), "shared_families_across_train_test": len(shared_families)})
display(pd.DataFrame(layout_rows))
print("Layout split can put related layout families on both sides. Treat it as a secondary interpolation protocol, not the primary generalization estimate.")

## Fold-size visualization and interpretation

Grouped folds are not necessarily exactly equal in size because complete groups cannot be split. This chart shows the row count per held-out fold and makes the family-level split structure visible.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
family_counts = fold_table.groupby("fold").agg(train_cases=("train_cases", "first"), test_cases=("test_cases", "first")) if {"train_cases", "test_cases"}.issubset(fold_table.columns) else None
if family_counts is not None:
    axes[0].bar(family_counts.index.astype(str), family_counts.test_cases)
    axes[0].set(title="Family-held-out test cases by fold", xlabel="Fold", ylabel="Test rows")
else:
    axes[0].text(.5, .5, "Fold manifest columns differ; inspect table above.", ha="center", va="center")
    axes[0].set_axis_off()
layout_table = pd.DataFrame(layout_rows)
if {"fold", "test_layouts"}.issubset(layout_table.columns):
    axes[1].bar(layout_table.fold.astype(str), layout_table.test_layouts)
    axes[1].set(title="Layout-held-out test layouts by fold", xlabel="Fold", ylabel="Test layouts")
else:
    axes[1].text(.5, .5, "See layout-fold table above.", ha="center", va="center")
    axes[1].set_axis_off()
plt.tight_layout(); plt.show()
print("Primary validation must keep every family wholly within train or test; the assertions above enforce this condition.")

## 3. Leakage-safe preprocessing example

Imputation, scaling and one-hot encoding are placed inside a pipeline. During each fold, `fit` is called on training rows only; the held-out rows are only transformed and predicted. This is important even when the current table has no missing values, because the same code should remain valid if future cases contain them.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
])
train_idx, test_idx = next(iter(cv.split(X, y, groups)))
X_train_transformed = preprocessor.fit_transform(X.iloc[train_idx])
X_test_transformed = preprocessor.transform(X.iloc[test_idx])
print("Train transformed shape:", X_train_transformed.shape)
print("Test transformed shape:", X_test_transformed.shape)
print("Preprocessor fit on training rows only; test rows were transformed without fitting.")

## 4. Save the fold manifest

In [ ]:
fold_path = RESULTS / "family_fold_manifest.csv"
fold_table.to_csv(fold_path, index=False)
print("Created:", fold_path)